# Quickstart: a reason-deletion certificate

> **Demonstration only:** this runs on reasonsmith's frozen synthetic example, not evidence about any real decision. Reasonsmith reports evidence and refusals; it does not certify compliance or provide legal advice.

The shipped `TruncatingCreditSystem` states one reason while its inference artefact used five. The certificate reruns the inference under reason deletions; it does not trust the reason text in the log. See [`docs/theory/07-explanation.md`](../docs/theory/07-explanation.md) for the deletion contract.

In [1]:
import subprocess

command = [
    "reasonsmith", "check",
    "--system-module", "reasonsmith.examples.truncating_credit_system:system_under_test",
    "--pack", "ecoa",
]
run = subprocess.run(command, text=True, capture_output=True, check=False)
assert run.returncode == 2  # the shipped example intentionally violates one duty
for line in run.stdout.splitlines():
    if ("headline:" in line or "principal_reasons_complete" in line or
            "On decision #1 exact inference" in line or "Measured against" in line):
        print(line)


headline: 6 requirements · 6 binding: 3 observed, 1 violated, 1 not evaluated, 1 unattainable · all positives observed-only
  [PROBED] ecoa_reg_b_1002_9_b_2_principal_reasons_complete (ECOA / Regulation B (12 CFR 1002.9) 12 CFR 1002.9(b)(2)): violated
    summary: Violated on 1 of 2 certified decision(s): the stated reasons are not all the reasons. On decision #1 exact inference found 5 reason(s) and the deletion probe showed the system's answer does not depend on 4 of them — C05 — Insufficient number of credit references provided; C03 — Delinquent past or present credit obligations; C04 — Too many recent inquiries on credit bureau report; C02 — Length of time credit has been established is too short. Attribution: The deleted reasons are exactly the 4 lowest-scoring of the 5, and the engine kept the top 1. This is the signature of top-k proof truncation at k=1: top-k works by discarding proofs, so the dropped reasons are lost by configuration, not by error. The missing probability mass

The four names below are a **measurement**: reasonsmith enumerates the artefact's reasons, switches their private facts off, and reruns the system. A log can only recount what the system chose to print; it cannot establish which facts its inference depended on. The probe is therefore a counterfactual measurement of the exposed artefact, bounded by its reported budget.

In [2]:
from dataclasses import replace

from reasonsmith.demo import deployed_credit_system
from reasonsmith.report import check_conformance
from reasonsmith.spec import load_pack

pack = load_pack("ecoa")
requirement = pack.get_requirement("ecoa_reg_b_1002_9_b_2_principal_reasons_complete")
one_duty = replace(pack, id="ecoa:principal-reasons", requirements=(requirement,))
report = check_conformance(deployed_credit_system(), one_duty)
result = report.to_dict()["results"][0]
print(result["verdict"], result["strength"])
failed = result["details"]["certificates"][1]
print("reasons found:", failed["reasons_found"])
print("reasons deleted:")
for reason in failed["missing_reasons"]:
    print(" -", reason)


violated probed
reasons found: 5
reasons deleted:
 - C05 — Insufficient number of credit references provided
 - C03 — Delinquent past or present credit obligations
 - C04 — Too many recent inquiries on credit bureau report
 - C02 — Length of time credit has been established is too short
